# FT-Transformer — Employee Resignation Prediction

**Member 1 model:** FT-Transformer  
**Task:** Binary classification  
**Target:** `Resigned` (`True` / `1` = resigned)  
**Primary metric:** Average Precision (reported as PR-AUC in project discussion)

## Objective

Train and evaluate FT-Transformer on the project's **80,000-row development partition** using the frozen **5-fold cross-validation** registry and all **17 approved predictors**.

The **20,000-row formal holdout is excluded** from training, preprocessing, tuning, model selection, and OOF evaluation. It remains reserved for the locked final evaluation stage.

## Methodology and Leakage Controls

Each outer CV round uses **64,000 training rows** and **16,000 validation rows**.

The FT-Transformer adapter owns fold-local preprocessing:

- 13 numerical/ordinal predictors are standardized with `StandardScaler` fitted on the outer-training folds only.
- 4 categorical predictors are encoded from mappings learned on the outer-training folds only.
- A fresh adapter is created for each outer fold.
- The outer validation fold is never passed to `fit()`.
- One probability is generated for every development employee, producing 80,000 OOF predictions.
- `Employee_ID`, `Hire_Date`, and `Resigned` are never used as predictors.

The shared repository runner also rejects any formal-holdout `Employee_ID` passed into OOF training.

In [ ]:
# -----------------------------
# 1. Imports and project setup
# -----------------------------

import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from sklearn.metrics import precision_recall_curve

# Resolve the repository root from notebooks/05_member_1/.
PROJECT_ROOT = Path("../..").resolve()

# Fallback for notebook launch from a different working directory.
if not (PROJECT_ROOT / "configs" / "feature_contract.yaml").exists():
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "configs" / "feature_contract.yaml").exists():
            PROJECT_ROOT = candidate.resolve()
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import calculate_binary_metrics
from src.models.ft_transformer import FTTransformerAdapter
from src.models.runner import run_5fold_cv

pd.set_option("display.max_columns", None)

CONFIG_PATH = PROJECT_ROOT / "configs" / "models" / "ft_transformer.yaml"
FEATURE_CONTRACT_PATH = PROJECT_ROOT / "configs" / "feature_contract.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Extended_Employee_Performance_and_Productivity_Data.csv"
SPLIT_PATH = PROJECT_ROOT / "data" / "processed" / "split_registry.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "models" / "ft_transformer"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [CONFIG_PATH, FEATURE_CONTRACT_PATH, DATA_PATH, SPLIT_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, f"Missing required project files: {missing_paths}"

print("Project root:", PROJECT_ROOT)
print("Results directory:", RESULTS_DIR)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# -----------------------------
# 2. Load configuration and record the executed parameter set
# -----------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    ft_config = yaml.safe_load(file)

with open(FEATURE_CONTRACT_PATH, "r", encoding="utf-8") as file:
    feature_contract = yaml.safe_load(file)

model_name = ft_config["model"]["name"]
model_id = ft_config["model"]["model_id"]

# Start from the fixed architecture and make optimizer defaults explicit.
fixed_params = ft_config["fixed_params"].copy()
fixed_params.setdefault(
    "learning_rate",
    float(ft_config["search_space"]["learning_rate"]["default"]),
)
fixed_params.setdefault(
    "weight_decay",
    float(ft_config["search_space"]["weight_decay"]["default"]),
)

config_summary = pd.DataFrame(
    [
        ("Model", model_name),
        ("Model ID", model_id),
        ("Implementation", ft_config["model"]["implementation"]),
        ("Seed", fixed_params["seed"]),
        ("Transformer blocks", fixed_params["n_blocks"]),
        ("Block dimension", fixed_params["d_block"]),
        ("Attention heads", fixed_params["attention_n_heads"]),
        ("FFN hidden dimension", fixed_params["ffn_d_hidden"]),
        ("Learning rate", fixed_params["learning_rate"]),
        ("Weight decay", fixed_params["weight_decay"]),
        ("Executed epochs per fold", 50),
        ("Batch size", 512),
    ],
    columns=["Setting", "Value"],
)

display(config_summary)

print("\nConfigured search-space parameters:")
display(
    pd.DataFrame(ft_config["search_space"])
    .T
    .rename_axis("parameter")
    .reset_index()
)

In [ ]:
# -----------------------------
# 3. Reproducibility and compute environment
# -----------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_label = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

print("Python kernel:", sys.executable)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Training device:", device_label)

if torch.cuda.is_available():
    print("PyTorch CUDA runtime:", torch.version.cuda)
    print(
        "GPU memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB",
    )

print("Random seed:", fixed_params["seed"])

In [ ]:
# -----------------------------
# 4. Load raw data and frozen split registry
# -----------------------------

df = pd.read_csv(DATA_PATH)
split_registry = pd.read_csv(SPLIT_PATH)

assert df.shape == (100_000, 20), f"Unexpected raw dataset shape: {df.shape}"
assert split_registry.shape == (100_000, 3), (
    f"Unexpected split-registry shape: {split_registry.shape}"
)
assert df["Employee_ID"].is_unique
assert split_registry["Employee_ID"].is_unique

print("Raw dataset shape:", df.shape)
print("Split registry shape:", split_registry.shape)
print("\nPartition counts:")
print(split_registry["partition"].value_counts())

In [ ]:
# -----------------------------
# 5. Isolate the development partition and enforce holdout protection
# -----------------------------

development_ids = set(
    split_registry.loc[
        split_registry["partition"] == "development",
        "Employee_ID",
    ]
)
holdout_ids = set(
    split_registry.loc[
        split_registry["partition"] == "holdout",
        "Employee_ID",
    ]
)

development_data = (
    df.loc[df["Employee_ID"].isin(development_ids)]
    .copy()
    .reset_index(drop=True)
)

development_df = development_data.merge(
    split_registry[["Employee_ID", "partition", "cv_fold"]],
    on="Employee_ID",
    how="left",
    validate="one_to_one",
)
development_df["cv_fold"] = development_df["cv_fold"].astype(int)

assert len(development_data) == 80_000
assert len(holdout_ids) == 20_000
assert set(development_data["Employee_ID"]).isdisjoint(holdout_ids)
assert set(development_df["cv_fold"].unique()) == {1, 2, 3, 4, 5}
assert development_df["cv_fold"].value_counts().eq(16_000).all()

print("Development rows:", len(development_data))
print("Formal holdout rows excluded:", len(holdout_ids))
print("Holdout IDs present in modeling data: 0")
print("\nDevelopment fold counts:")
print(development_df["cv_fold"].value_counts().sort_index())

In [ ]:
# -----------------------------
# 6. Validate the feature contract and build X / y
# -----------------------------

numerical_features = feature_contract["numerical_features"]
categorical_features = feature_contract["categorical_features"]
primary_model_features = feature_contract["primary_model_features"]
target_config = feature_contract["target"]

numerical_feature_names = [item["name"] for item in numerical_features]
categorical_feature_names = [item["name"] for item in categorical_features]
target_column = target_config["column"]

assert len(numerical_feature_names) == 13
assert len(categorical_feature_names) == 4
assert len(primary_model_features) == 17
assert set(numerical_feature_names + categorical_feature_names) == set(primary_model_features)

required_columns = primary_model_features + [target_column, "Employee_ID", "cv_fold"]
missing_columns = [c for c in required_columns if c not in development_df.columns]
assert not missing_columns, f"Missing required columns: {missing_columns}"

X = development_df[primary_model_features].copy()
y = development_df[target_column].astype(int).copy()
employee_ids = development_df["Employee_ID"].copy()
cv_folds = development_df["cv_fold"].copy()

assert X.shape == (80_000, 17)
assert len(y) == 80_000
assert set(y.unique()) == {0, 1}
assert X.isna().sum().sum() == 0
assert "Employee_ID" not in X.columns
assert "Hire_Date" not in X.columns
assert target_column not in X.columns

print("Predictor matrix:", X.shape)
print("Numerical / ordinal predictors:", len(numerical_feature_names))
print("Categorical predictors:", len(categorical_feature_names))
print("Missing predictor values:", int(X.isna().sum().sum()))
print("Development resignation prevalence:", f"{y.mean():.4%}")

In [ ]:
# -----------------------------
# 7. Verify fold balance
# -----------------------------

fold_summary = (
    development_df
    .groupby("cv_fold")[target_column]
    .agg(
        total_rows="size",
        resigned_count="sum",
        resignation_rate="mean",
    )
    .reset_index()
)

fold_summary["not_resigned_count"] = (
    fold_summary["total_rows"] - fold_summary["resigned_count"]
)
fold_summary["resignation_rate_pct"] = (
    fold_summary["resignation_rate"] * 100
).round(3)

fold_summary = fold_summary[
    [
        "cv_fold",
        "total_rows",
        "not_resigned_count",
        "resigned_count",
        "resignation_rate_pct",
    ]
]

assert len(fold_summary) == 5
assert (fold_summary["total_rows"] == 16_000).all()
assert (fold_summary["resigned_count"] > 0).all()
assert (fold_summary["not_resigned_count"] > 0).all()

display(fold_summary)

In [ ]:
# -----------------------------
# 8. Verify categorical coverage across outer folds
# -----------------------------

coverage_records = []

for fold in sorted(development_df["cv_fold"].unique()):
    train_fold_df = development_df[development_df["cv_fold"] != fold]
    val_fold_df = development_df[development_df["cv_fold"] == fold]

    for feature in categorical_feature_names:
        train_categories = set(train_fold_df[feature].dropna().unique())
        validation_categories = set(val_fold_df[feature].dropna().unique())
        unseen = validation_categories - train_categories

        coverage_records.append(
            {
                "cv_fold": fold,
                "feature": feature,
                "train_categories": len(train_categories),
                "validation_categories": len(validation_categories),
                "unseen_categories": len(unseen),
            }
        )

categorical_coverage = pd.DataFrame(coverage_records)
total_unseen = int(categorical_coverage["unseen_categories"].sum())

assert total_unseen == 0, (
    "At least one outer-validation fold contains a category absent "
    "from its corresponding training folds."
)

display(categorical_coverage)
print("\nTotal unseen validation categories:", total_unseen)

## Verified FT-Transformer Execution Protocol

The repository implementation was inspected before training.

**Executed behavior**

- Fresh `FTTransformerAdapter` per outer fold.
- Fold-local `StandardScaler` and categorical mappings.
- AdamW optimizer.
- `BCEWithLogitsLoss`.
- 50 epochs per fold.
- Batch size 512.
- Learning rate 0.001.
- Weight decay 1e-5.
- Seed 42.
- Automatic device selection (`cuda` when available, otherwise CPU).

**Configuration caveat**

The YAML also declares `max_epochs: 100`, `patience: 15`, `ReduceLROnPlateau`, and an inner validation fraction of 0.15. The current shared runner calls `adapter.fit(X, y)` without supplying those settings, and the current FT-Transformer adapter does not implement inner-validation early stopping or the scheduler. This notebook therefore reports the behavior that is **actually executed**, not the unused YAML declarations.

The YAML search space is recorded for reproducibility, but this completed run uses the fixed/default configuration; it is **not presented as a completed hyperparameter-search result**.

The source adapter also passes `ffn_d_hidden_multiplier=None` for compatibility with `rtdl_revisiting_models==0.0.2` while using the explicit `ffn_d_hidden` value.

In [ ]:
# -----------------------------
# 9. Create a fresh adapter factory
# -----------------------------

def make_ft_transformer():
    """Return a fresh, unfitted FT-Transformer adapter."""
    return FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=fixed_params.copy(),
        categorical_features=categorical_feature_names.copy(),
    )

adapter_a = make_ft_transformer()
adapter_b = make_ft_transformer()

assert adapter_a is not adapter_b
assert adapter_a.params is not adapter_b.params
assert adapter_a.categorical_features is not adapter_b.categorical_features
assert not adapter_a.get_metadata()["fitted"]
assert not adapter_b.get_metadata()["fitted"]

print("Adapter factory validation passed.")
print("Model:", adapter_a.name)
print("Categorical predictors:", adapter_a.categorical_features)

In [ ]:
# -----------------------------
# 10. Official 5-fold OOF training
# -----------------------------

print("FT-Transformer official CV training")
print("-----------------------------------")
print("Development rows:", len(development_data))
print("Formal holdout rows used: 0")
print("Predictors:", len(primary_model_features))
print("Outer CV folds: 5")
print("Training rows per fold: 64,000")
print("Validation rows per fold: 16,000")
print("Epochs per fold: 50")
print("Batch size: 512")
print("Training device:", device_label)

cv_start_time = time.perf_counter()

ft_cv_result = run_5fold_cv(
    adapter_factory=make_ft_transformer,
    data=development_data,
    split_registry=split_registry,
    feature_cols=primary_model_features,
    target_col=target_column,
    id_col="Employee_ID",
    fold_col="cv_fold",
    partition_col="partition",
    development_label="development",
    holdout_label="holdout",
    expected_folds=(1, 2, 3, 4, 5),
    expected_model_name=model_name,
)

cv_elapsed_time = time.perf_counter() - cv_start_time

print("\nFT-Transformer 5-fold CV completed successfully.")
print("Wall-clock time:", f"{cv_elapsed_time / 60:.2f} minutes")
print("OOF prediction rows:", len(ft_cv_result.predictions))

### Recorded Successful Run

The uploaded working notebook already completed the official 5-fold development run successfully. The recorded result was:

| Metric | Recorded value |
|---|---:|
| Pooled Average Precision / PR-AUC | **0.101132** |
| Pooled ROC-AUC | **0.498821** |
| Pooled Brier score | **0.090651** |
| Pooled F1 @ 0.50 | **0.000000** |
| CV mean Average Precision | **0.102028** |
| CV SD Average Precision | **0.002378** |
| CV mean ROC-AUC | **0.499769** |
| CV SD ROC-AUC | **0.010585** |
| Total fit time | **770.90 s** |
| Total prediction time | **1.16 s** |
| Wall-clock time reported by notebook | **12.88 min** |

CUDA was verified as available with an **NVIDIA GeForce RTX 4060 Laptop GPU (8 GB)** before this run. The earlier `"Training device: CPU"` text in the development notebook was a hard-coded print statement; the adapter itself selects CUDA dynamically when `torch.cuda.is_available()` is `True`.

In [ ]:
# -----------------------------
# 11. Fold-level performance
# -----------------------------

fold_metrics = ft_cv_result.fold_metrics.copy()

expected_metric_columns = {
    "fold",
    "average_precision",
    "roc_auc",
    "precision",
    "recall",
    "f1",
    "balanced_accuracy",
    "brier_score",
    "fit_time_seconds",
    "predict_time_seconds",
}
missing_metric_columns = expected_metric_columns - set(fold_metrics.columns)
assert not missing_metric_columns, (
    f"Fold metrics are missing columns: {sorted(missing_metric_columns)}"
)
assert len(fold_metrics) == 5

display(
    fold_metrics[
        [
            "fold",
            "average_precision",
            "roc_auc",
            "brier_score",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
            "fit_time_seconds",
            "predict_time_seconds",
        ]
    ].round(6)
)

In [ ]:
# -----------------------------
# 12. Aggregate OOF metrics and no-skill reference
# -----------------------------

oof_predictions = ft_cv_result.predictions.copy()

assert len(oof_predictions) == 80_000
assert oof_predictions["Employee_ID"].is_unique
assert oof_predictions["y_probability"].between(0, 1).all()
assert oof_predictions["y_probability"].notna().all()
assert set(oof_predictions["cv_fold"].unique()) == {1, 2, 3, 4, 5}
assert set(oof_predictions["Employee_ID"]).isdisjoint(holdout_ids)

pooled_metrics = calculate_binary_metrics(
    y_true=oof_predictions["y_true"],
    y_probability=oof_predictions["y_probability"],
    y_pred=oof_predictions["y_pred_0_5"],
    threshold=0.50,
)

prevalence = float(oof_predictions["y_true"].mean())
no_skill_ap = prevalence
constant_prevalence_brier = prevalence * (1.0 - prevalence)

metric_summary = pd.DataFrame(
    [
        ("Average Precision / PR-AUC", pooled_metrics["average_precision"]),
        ("No-skill AP (prevalence)", no_skill_ap),
        ("ROC-AUC", pooled_metrics["roc_auc"]),
        ("Brier score", pooled_metrics["brier_score"]),
        ("Constant-prevalence Brier reference", constant_prevalence_brier),
        ("Balanced accuracy @ 0.50", pooled_metrics["balanced_accuracy"]),
        ("Precision @ 0.50", pooled_metrics["precision"]),
        ("Recall @ 0.50", pooled_metrics["recall"]),
        ("F1 @ 0.50", pooled_metrics["f1"]),
        ("Predicted-positive rate @ 0.50", pooled_metrics["predicted_positive_rate"]),
    ],
    columns=["Metric", "Value"],
)

display(metric_summary.round(6))

print(
    "\nPR-AUC lift over prevalence:",
    f"{pooled_metrics['average_precision'] - no_skill_ap:+.6f}",
)

In [ ]:
# -----------------------------
# 13. OOF probability diagnostics
# -----------------------------

probabilities = oof_predictions["y_probability"]

probability_summary = pd.DataFrame(
    {
        "Statistic": [
            "Minimum",
            "1st percentile",
            "25th percentile",
            "Median",
            "Mean",
            "75th percentile",
            "99th percentile",
            "Maximum",
        ],
        "Probability": [
            probabilities.min(),
            probabilities.quantile(0.01),
            probabilities.quantile(0.25),
            probabilities.median(),
            probabilities.mean(),
            probabilities.quantile(0.75),
            probabilities.quantile(0.99),
            probabilities.max(),
        ],
    }
)

display(probability_summary.round(6))

print(
    "Predicted positives at threshold 0.50:",
    int(oof_predictions["y_pred_0_5"].sum()),
)
print(
    "Actual positives:",
    int(oof_predictions["y_true"].sum()),
)

In [ ]:
# -----------------------------
# 14. Precision-recall curve and probability distribution
# -----------------------------

precision, recall, _ = precision_recall_curve(
    oof_predictions["y_true"],
    oof_predictions["y_probability"],
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall, precision, label=f"FT-Transformer AP = {pooled_metrics['average_precision']:.4f}")
ax.axhline(prevalence, linestyle="--", label=f"No-skill prevalence = {prevalence:.4f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("FT-Transformer OOF Precision-Recall Curve")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()

pr_curve_path = RESULTS_DIR / "figure_01_ft_transformer_precision_recall_curve.png"
fig.savefig(pr_curve_path, dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(
    [
        oof_predictions.loc[oof_predictions["y_true"] == 0, "y_probability"],
        oof_predictions.loc[oof_predictions["y_true"] == 1, "y_probability"],
    ],
    bins=40,
    label=["Not resigned", "Resigned"],
    alpha=0.7,
)
ax.set_xlabel("Predicted resignation probability")
ax.set_ylabel("Employees")
ax.set_title("FT-Transformer OOF Probability Distribution")
ax.legend()
fig.tight_layout()

probability_plot_path = RESULTS_DIR / "figure_02_ft_transformer_oof_probability_distribution.png"
fig.savefig(probability_plot_path, dpi=150, bbox_inches="tight")
plt.show()

## Tuning, Interpretability, and Serialization Scope

### Hyperparameter tuning
The YAML contains a search space, but the recorded run above evaluates the fixed/default configuration. It should **not** be described as a tuned optimum. Any later tuning must remain development-only and reuse the frozen folds without accessing the formal holdout.

### Interpretability
The current FT-Transformer adapter does not expose stable per-feature importance, attention attribution, SHAP values, or retained fold models through the shared OOF runner. This notebook therefore does **not fabricate a feature-importance ranking**. Cross-model interpretability and fairness analyses should use methods that are explicitly supported by the selected model implementation.

### Model serialization
The shared OOF runner returns predictions, metrics, timing, and metadata, but it does not retain all five fitted fold adapters. This notebook therefore serializes the **evaluation artifacts** needed for model comparison. Final deployable-model training and model-weight serialization should occur only after model selection and pipeline freeze.

In [ ]:
# -----------------------------
# 15. Save reproducible FT-Transformer evaluation artifacts
# -----------------------------

predictions_path = RESULTS_DIR / "ft_transformer_oof_predictions.csv"
fold_metrics_path = RESULTS_DIR / "ft_transformer_fold_metrics.csv"
summary_metrics_path = RESULTS_DIR / "ft_transformer_summary_metrics.json"
run_metadata_path = RESULTS_DIR / "ft_transformer_run_metadata.json"
manifest_path = RESULTS_DIR / "ft_transformer_output_manifest.csv"

oof_predictions.to_csv(predictions_path, index=False)
fold_metrics.to_csv(fold_metrics_path, index=False)

with open(summary_metrics_path, "w", encoding="utf-8") as file:
    json.dump(ft_cv_result.summary_metrics, file, indent=2)

run_metadata = {
    "model_name": model_name,
    "model_id": model_id,
    "feature_count": len(primary_model_features),
    "features": primary_model_features,
    "categorical_features": categorical_feature_names,
    "numerical_features": numerical_feature_names,
    "development_rows": len(development_data),
    "formal_holdout_rows_used": 0,
    "outer_folds": [1, 2, 3, 4, 5],
    "epochs_per_fold": 50,
    "batch_size": 512,
    "device": device_label,
    "torch_version": torch.__version__,
    "torch_cuda_runtime": torch.version.cuda,
    "cuda_available": bool(torch.cuda.is_available()),
    "executed_params": fixed_params,
    "tuning_performed": False,
    "adapter_metadata": ft_cv_result.metadata,
    "fit_time_seconds_total": ft_cv_result.fit_time_seconds_total,
    "predict_time_seconds_total": ft_cv_result.predict_time_seconds_total,
}

with open(run_metadata_path, "w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)

artifact_paths = [
    predictions_path,
    fold_metrics_path,
    summary_metrics_path,
    run_metadata_path,
    pr_curve_path,
    probability_plot_path,
]

manifest = pd.DataFrame(
    {
        "artifact": [path.name for path in artifact_paths],
        "relative_path": [
            str(path.relative_to(PROJECT_ROOT))
            for path in artifact_paths
        ],
        "exists": [path.exists() for path in artifact_paths],
    }
)
manifest.to_csv(manifest_path, index=False)

display(manifest)
print("\nArtifacts saved to:", RESULTS_DIR)

In [ ]:
# -----------------------------
# 16. Final integrity checks
# -----------------------------

assert len(oof_predictions) == 80_000
assert oof_predictions["Employee_ID"].nunique() == 80_000
assert set(oof_predictions["Employee_ID"]).isdisjoint(holdout_ids)
assert oof_predictions["y_probability"].between(0, 1).all()
assert oof_predictions["y_probability"].notna().all()
assert len(fold_metrics) == 5
assert set(fold_metrics["fold"]) == {1, 2, 3, 4, 5}

assert np.isclose(
    pooled_metrics["average_precision"],
    ft_cv_result.summary_metrics["pooled_average_precision"],
)
assert np.isclose(
    pooled_metrics["roc_auc"],
    ft_cv_result.summary_metrics["pooled_roc_auc"],
)
assert np.isclose(
    pooled_metrics["brier_score"],
    ft_cv_result.summary_metrics["pooled_brier_score"],
)

required_artifacts = artifact_paths + [manifest_path]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
assert not missing_artifacts, f"Missing result artifacts: {missing_artifacts}"

print("All FT-Transformer final integrity checks passed.")
print("Development OOF evaluation is ready for model comparison.")
print("Formal holdout remains untouched.")

## Conclusion

FT-Transformer was evaluated on all **17 approved predictors** using the frozen five-fold development protocol. The formal 20,000-row holdout was not used.

The recorded completed run achieved:

- **Pooled PR-AUC / Average Precision:** 0.101132
- **Pooled ROC-AUC:** 0.498821
- **Pooled Brier score:** 0.090651
- **Pooled F1 at threshold 0.50:** 0.000000

Development-set resignation prevalence is approximately **10.01%**, so the model's PR-AUC is essentially at the no-skill prevalence reference. ROC-AUC is also approximately 0.50, indicating little to no useful ranking discrimination under this configuration. The zero F1 at 0.50 is consistent with probabilities remaining below the default classification threshold.

This is a valid negative modeling result: the current FT-Transformer configuration does not demonstrate meaningful predictive signal on the approved feature set. The OOF artifacts should proceed to the common model-comparison stage, while the formal holdout remains locked.

**Status:** FT-Transformer development OOF evaluation complete; ready for cross-model comparison. Hyperparameter tuning is not claimed as complete.